# Sentiment Analysis

In [2]:
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --------------------------------------------------
# 1. Define the two Hugging Face models
# --------------------------------------------------

MODEL_NAMES = {
    "financial_roberta": "soleimanian/financial-roberta-large-sentiment",
    "finbert": "ProsusAI/finbert",
}

device = torch.device("cpu")

models = {}
tokenizers = {}
model_labels = {}
chunk_sizes = {}

EXPECTED_LABELS = {"negative", "neutral", "positive"}

# --------------------------------------------------
# 2. Load the models and verify their labels
# --------------------------------------------------

for model_key, model_name in MODEL_NAMES.items():

    print(f"Loading {model_key}: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    model.to(device)
    model.eval()

    # Read the model's label order
    labels = [
        str(model.config.id2label[i]).lower()
        for i in range(model.config.num_labels)
    ]

    # Standardize common label names
    label_mapping = {
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
        "neg": "negative",
        "neu": "neutral",
        "pos": "positive",
    }

    labels = [label_mapping.get(label, label) for label in labels]

    if len(labels) != 3 or set(labels) != EXPECTED_LABELS:
        raise ValueError(
            f"{model_key} does not have the expected three classes. "
            f"Found labels: {labels}. Check the model card."
        )

    # Determine the maximum supported input length
    configured_limit = getattr(
        model.config, "max_position_embeddings", 512
    )
    tokenizer_limit = tokenizer.model_max_length

    if tokenizer_limit > 100_000:
        tokenizer_limit = configured_limit

    max_length = min(tokenizer_limit, configured_limit)

    chunk_size = (
        max_length
        - tokenizer.num_special_tokens_to_add(pair=False)
    )

    models[model_key] = model
    tokenizers[model_key] = tokenizer
    model_labels[model_key] = labels
    chunk_sizes[model_key] = chunk_size

    print("Labels:", labels)
    print("Maximum length:", max_length)
    print("Chunk size:", chunk_size)
    print("-" * 50)

print("Both models loaded successfully.")


Loading financial_roberta: soleimanian/financial-roberta-large-sentiment
Labels: ['negative', 'neutral', 'positive']
Maximum length: 514
Chunk size: 512
--------------------------------------------------
Loading finbert: ProsusAI/finbert
Labels: ['positive', 'negative', 'neutral']
Maximum length: 512
Chunk size: 510
--------------------------------------------------
Both models loaded successfully.


In [3]:
def predict_sentiment(text, model_key, batch_size=16):

    # Handle missing or empty paragraphs
    if pd.isna(text) or not str(text).strip():
        return {
            "sentiment": None,
            "positive_prob": np.nan,
            "neutral_prob": np.nan,
            "negative_prob": np.nan,
            "n_chunks": 0,
        }

    tokenizer = tokenizers[model_key]
    model = models[model_key]
    labels = model_labels[model_key]
    chunk_size = chunk_sizes[model_key]

    # Tokenize without truncating the paragraph
    token_ids = tokenizer(
        str(text),
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    # Split into model-compatible chunks
    chunks = [
        token_ids[i:i + chunk_size]
        for i in range(0, len(token_ids), chunk_size)
    ]

    if not chunks:
        return {
            "sentiment": None,
            "positive_prob": np.nan,
            "neutral_prob": np.nan,
            "negative_prob": np.nan,
            "n_chunks": 0,
        }

    all_probabilities = []

    for start in range(0, len(chunks), batch_size):

        batch_chunks = chunks[start:start + batch_size]

        encoded_chunks = [
            tokenizer.prepare_for_model(
                chunk,
                add_special_tokens=True,
                truncation=False,
                return_attention_mask=True,
            )
            for chunk in batch_chunks
        ]

        batch_inputs = tokenizer.pad(
            encoded_chunks,
            padding=True,
            return_tensors="pt",
        )

        with torch.no_grad():
            logits = model(**batch_inputs).logits
            probabilities = torch.softmax(logits, dim=1)

        all_probabilities.extend(probabilities.cpu().numpy())

    # Average probabilities across chunks
    mean_probs = np.mean(all_probabilities, axis=0)

    probs = {
        label: float(mean_probs[i])
        for i, label in enumerate(labels)
    }

    sentiment = max(probs, key=probs.get)

    return {
        "sentiment": sentiment,
        "positive_prob": probs["positive"],
        "neutral_prob": probs["neutral"],
        "negative_prob": probs["negative"],
        "n_chunks": len(chunks),
    }


In [4]:
import pandas as pd
import openpyxl


import pandas as pd
import openpyxl

# 1. Load the correct Excel file for each company
data_adidas = pd.read_excel("../data/preparation/adidas.xlsx", sheet_name="Sheet1")
data_puma = pd.read_excel("../data/preparation/puma.xlsx", sheet_name="Sheet1")
data_asics = pd.read_excel("../data/preparation/asics.xlsx", sheet_name="Sheet1")
data_mizuno = pd.read_excel("../data/preparation/mizuno.xlsx", sheet_name="Sheet1")


datasets = {
    "adidas": data_adidas,
    "puma": data_puma,
    "asics": data_asics,
    "mizuno": data_mizuno,
}

results = {}

for company, data in datasets.items():

    df = data.copy()

    # Apply both models to every paragraph
    for model_key in MODEL_NAMES:

        predictions = df["Text"].apply(
            lambda text: predict_sentiment(text, model_key)
        )

        prediction_df = pd.DataFrame(
            predictions.tolist(),
            index=df.index,
        )

        # Add a prefix to identify the model
        prediction_df = prediction_df.add_prefix(
            f"{model_key}_"
        )

        df = pd.concat(
            [df, prediction_df],
            axis=1,
        )

    results[company] = df

    print(f"{company}: {len(df)} paragraphs analyzed")


You're using a RobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


adidas: 219 paragraphs analyzed
puma: 105 paragraphs analyzed
asics: 206 paragraphs analyzed
mizuno: 62 paragraphs analyzed


## Compare the sentiment distributions

In [5]:
summary_rows = []

for company, df in results.items():

    for model_key in MODEL_NAMES:

        sentiment_col = f"{model_key}_sentiment"

        proportions = (
            df[sentiment_col]
            .dropna()
            .value_counts(normalize=True)
            .reindex(
                ["negative", "neutral", "positive"],
                fill_value=0,
            )
            .mul(100)
        )

        for sentiment, percentage in proportions.items():

            summary_rows.append({
                "company": company,
                "model": model_key,
                "sentiment": sentiment,
                "percentage": percentage,
            })

sentiment_summary = pd.DataFrame(summary_rows)

comparison = sentiment_summary.pivot_table(
    index=["company", "sentiment"],
    columns="model",
    values="percentage",
)

print(comparison.round(2))


model              financial_roberta  finbert
company sentiment                            
adidas  negative                3.65     2.28
        neutral                39.73    71.23
        positive               56.62    26.48
asics   negative                2.91     1.46
        neutral                32.52    59.22
        positive               64.56    39.32
mizuno  negative                3.23     0.00
        neutral                33.87    59.68
        positive               62.90    40.32
puma    negative               11.43    11.43
        neutral                25.71    21.90
        positive               62.86    66.67


In [6]:

from pathlib import Path

output_dir = Path("../data/input")
output_dir.mkdir(parents=True, exist_ok=True)

for company, df in results.items():

    output_path = output_dir / f"{company}_sentiment.xlsx"

    df.to_excel(output_path, index=False)

    print(f"Saved: {output_path}")

sentiment_summary.to_excel(
    output_dir / "sentiment_summary.xlsx",
    index=False,
)

print("All sentiment results saved.")


Saved: ../data/input/adidas_sentiment.xlsx
Saved: ../data/input/puma_sentiment.xlsx
Saved: ../data/input/asics_sentiment.xlsx
Saved: ../data/input/mizuno_sentiment.xlsx
All sentiment results saved.
